## Vector Stores and Retrievers
Langchain as Vector stores and retriever abstraction to fetch data for the model inference

In [1]:
## Documents - which are used as source
from langchain_core.documents import Document
documnets=[
    Document(
        page_content="Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth",
        metadata={"source":"photosynthesis-doc"}
    ),
    Document(
        page_content="he reaction, occurring in two stages light-dependent and Calvin cycle, uses sunlight to break down, release oxygen as a byproduct",
        metadata={"source":"photosynthesis-doc"}
    ),
    Document(
        page_content="Significance: Produces food for plants and oxygen for animals and humans.",
        metadata={"source":"photosynthesis-doc"}
    )
]

In [2]:
documnets

[Document(metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth'),
 Document(metadata={'source': 'photosynthesis-doc'}, page_content='he reaction, occurring in two stages light-dependent and Calvin cycle, uses sunlight to break down, release oxygen as a byproduct'),
 Document(metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.')]

In [ ]:
## set the Model environment
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

/home/hp/Documents/GenAI/LangChain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
# model

from langchain_ollama import ChatOllama
llm=ChatOllama(model="llama3.2:3b")
llm

ChatOllama(model='llama3.2:3b')

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1282.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [7]:
##run this if you are using ubuntu system and facing sqllite version issue
# --- STEP 1: THE FIX (MUST BE FIRST) ---
import sys
__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

# --- STEP 2: VERIFY ---
import sqlite3

print(f"Current SQLite version: {sqlite3.sqlite_version}")

Current SQLite version: 3.51.1


In [8]:
from langchain_chroma import Chroma
vectorstore= Chroma.from_documents(documents=documnets,embedding=embeddings)
vectorstore

In [10]:
vectorstore.similarity_search("photosynthetis")

[Document(id='7b589f47-a04f-48ed-b409-66a92a8343fa', metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth'),
 Document(id='2c4e3dc4-5912-4a10-ba6e-2c96d7fd0def', metadata={'source': 'photosynthesis-doc'}, page_content='he reaction, occurring in two stages light-dependent and Calvin cycle, uses sunlight to break down, release oxygen as a byproduct'),
 Document(id='932ed257-1a98-4761-89b9-6c8b5e54f4a8', metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.')]

In [ ]:
## Async query -- telling the python not to wait until the response arrives, go and perform other tasks
await vectorstore.asimilarity_search("photosynthesis")

[Document(id='7b589f47-a04f-48ed-b409-66a92a8343fa', metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth'),
 Document(id='2c4e3dc4-5912-4a10-ba6e-2c96d7fd0def', metadata={'source': 'photosynthesis-doc'}, page_content='he reaction, occurring in two stages light-dependent and Calvin cycle, uses sunlight to break down, release oxygen as a byproduct'),
 Document(id='932ed257-1a98-4761-89b9-6c8b5e54f4a8', metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.')]

In [12]:
vectorstore.similarity_search_with_score("photosynthetis")

[(Document(id='7b589f47-a04f-48ed-b409-66a92a8343fa', metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth'),
  1.0350284576416016),
 (Document(id='2c4e3dc4-5912-4a10-ba6e-2c96d7fd0def', metadata={'source': 'photosynthesis-doc'}, page_content='he reaction, occurring in two stages light-dependent and Calvin cycle, uses sunlight to break down, release oxygen as a byproduct'),
  1.1534688472747803),
 (Document(id='932ed257-1a98-4761-89b9-6c8b5e54f4a8', metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.'),
  1.5753508806228638)]

In [ ]:
## Retrievers -> LangChain runnables to chain the functions 
from langchain_core.runnables import RunnableLambda

retriever=RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["Photosynthesis","food"])

[[Document(id='7b589f47-a04f-48ed-b409-66a92a8343fa', metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth')],
 [Document(id='932ed257-1a98-4761-89b9-6c8b5e54f4a8', metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.')]]

In [16]:
# use the asretreiver functionality of vectorstore
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["photosynthesis","food"])

[[Document(id='7b589f47-a04f-48ed-b409-66a92a8343fa', metadata={'source': 'photosynthesis-doc'}, page_content='Photosynthesis is the essential biological process by which green plants, algae, and some bacteria convert light energy into chemical energy to fuel their growth')],
 [Document(id='932ed257-1a98-4761-89b9-6c8b5e54f4a8', metadata={'source': 'photosynthesis-doc'}, page_content='Significance: Produces food for plants and oxygen for animals and humans.')]]

In [23]:
## RAG pipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
message="""
Answer the questions based on the context only
{question}
Context:
{context}
 """
prompt=ChatPromptTemplate.from_messages(
    [("human",message)]
)

rag_chain= {"context":retriever,"question":RunnablePassthrough()}|prompt|llm
response=rag_chain.invoke("tell me how plants prepare food")
response.content

'Plants prepare food through a process called photosynthesis.'